In [1]:
# Cell 1: Stratified Block-Wise Train-Test Splitting for Distributed Network Nodes

import os
import numpy as np
import pandas as pd

# =============================================================================
# Configuration & Parameters
# =============================================================================
NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
BASE_INPUT_DIR = "dataset"
OUTPUT_DIR = "dataset/normalized"

BLOCK_SIZE = 1000
TRAIN_RATIO = 0.7  # 70% training data, 30% testing data per block
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# Execution: Block-Wise Shuffling and Splitting
# =============================================================================
for node in NODES:
    input_path = os.path.join(BASE_INPUT_DIR, f"Node_{node}_final_synthetic_dataset_with_source.csv")
    print(f"Processing node dataset: {input_path}")
    
    df = pd.read_csv(input_path)
    train_parts = []
    test_parts = []
    
    # Iterate through blocks to preserve identical temporal block distributions in train and test sets
    for i in range(0, len(df), BLOCK_SIZE):
        block = df.iloc[i:i + BLOCK_SIZE].copy()
        # Shuffle rows within the block to avoid sequence bias
        block = block.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
        
        n_train = int(len(block) * TRAIN_RATIO)
        train_parts.append(block.iloc[:n_train])
        test_parts.append(block.iloc[n_train:])
    
    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    
    print(f"-> Node {node} split complete | Train rows: {len(train_df)}, Test rows: {len(test_df)}")
    
    # Save split outputs
    train_out_path = os.path.join(OUTPUT_DIR, f"Node_{node}_train.csv")
    test_out_path = os.path.join(OUTPUT_DIR, f"Node_{node}_test.csv")
    
    train_df.to_csv(train_out_path, index=False)
    test_df.to_csv(test_out_path, index=False)

print(f"\nAll node splits successfully saved to: {OUTPUT_DIR}")

Processing node dataset: dataset/Node_A_final_synthetic_dataset_with_source.csv
-> Node A split complete | Train rows: 56000, Test rows: 24000
Processing node dataset: dataset/Node_B_final_synthetic_dataset_with_source.csv
-> Node B split complete | Train rows: 56000, Test rows: 24000
Processing node dataset: dataset/Node_C_final_synthetic_dataset_with_source.csv
-> Node C split complete | Train rows: 56000, Test rows: 24000
Processing node dataset: dataset/Node_D_final_synthetic_dataset_with_source.csv
-> Node D split complete | Train rows: 56000, Test rows: 24000
Processing node dataset: dataset/Node_E_final_synthetic_dataset_with_source.csv
-> Node E split complete | Train rows: 56000, Test rows: 24000
Processing node dataset: dataset/Node_F_final_synthetic_dataset_with_source.csv
-> Node F split complete | Train rows: 56000, Test rows: 24000
Processing node dataset: dataset/Node_G_final_synthetic_dataset_with_source.csv
-> Node G split complete | Train rows: 56000, Test rows: 24000

In [4]:
paper = "Paper-Methodik (Absatz für das Manuskript): Zur Vorbereitung der Klassifikation und Modellerevaluation wurden die knotenspezifischen Datensätze in Trainings- (70 %) und Testmengen (30 %) unterteilt. Um zeitliche Muster und die blockweise Zusammensetzung der synthetischen Injektionsszenarien konsistent über beide Teilmengen hinweg zu erhalten, erfolgte das Shuffling und Splitting blockweise (Blockgröße 1000). Dadurch wird eine identische Verteilung der Angriffsszenarien in Training und Test gewährleistet, was die Grundlage für eine verzerrungsfreie Validierung der verteilten Detektionsmodelle bildet."

print(paper)

Paper-Methodik (Absatz für das Manuskript): Zur Vorbereitung der Klassifikation und Modellerevaluation wurden die knotenspezifischen Datensätze in Trainings- (70 %) und Testmengen (30 %) unterteilt. Um zeitliche Muster und die blockweise Zusammensetzung der synthetischen Injektionsszenarien konsistent über beide Teilmengen hinweg zu erhalten, erfolgte das Shuffling und Splitting blockweise (Blockgröße 1000). Dadurch wird eine identische Verteilung der Angriffsszenarien in Training und Test gewährleistet, was die Grundlage für eine verzerrungsfreie Validierung der verteilten Detektionsmodelle bildet.


In [3]:
# Cell 2: Global Feature Normalization across all Nodes (Single Global Scaler)

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# =============================================================================
# Configuration & Parameters
# =============================================================================
NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
BASE_DIR = "dataset/normalized"
NUMERIC_COLS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
MODELS_DIR = "models"

os.makedirs(MODELS_DIR, exist_ok=True)

print("=== STEP 1: Fitting Global MinMaxScaler on Combined Training Data of All Nodes ===")

# 1. Load all train splits to fit a single unified global scaler (prevents data leakage per node)
train_dfs = [pd.read_csv(os.path.join(BASE_DIR, f"Node_{node}_train.csv")) for node in NODES]
combined_train_df = pd.concat(train_dfs, ignore_index=True)

global_scaler = MinMaxScaler(feature_range=(0, 1))
global_scaler.fit(combined_train_df[NUMERIC_COLS])

# Save global scaler for reusability during inference or evaluation
scaler_path = os.path.join(MODELS_DIR, "global_minmax_scaler.joblib")
joblib.dump(global_scaler, scaler_path)

print(f"Global Scaler successfully fitted and saved to: {scaler_path}")
print("Global Min values:", dict(zip(NUMERIC_COLS, global_scaler.data_min_)))
print("Global Max values:", dict(zip(NUMERIC_COLS, global_scaler.data_max_)))

print("\n=== STEP 2: Transforming and Saving Per-Node Splits Using the Global Scaler ===")

# 2. Apply the exact same global scaler to each individual node's train and test set
for node in NODES:
    train_path = os.path.join(BASE_DIR, f"Node_{node}_train.csv")
    test_path = os.path.join(BASE_DIR, f"Node_{node}_test.csv")
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    train_df_norm = train_df.copy()
    test_df_norm = test_df.copy()
    
    train_df_norm[NUMERIC_COLS] = global_scaler.transform(train_df[NUMERIC_COLS])
    test_df_norm[NUMERIC_COLS] = global_scaler.transform(test_df[NUMERIC_COLS])
    
    # Save globally normalized outputs
    norm_train_path = os.path.join(BASE_DIR, f"Node_{node}_train_normalized.csv")
    norm_test_path = os.path.join(BASE_DIR, f"Node_{node}_test_normalized.csv")
    
    train_df_norm.to_csv(norm_train_path, index=False)
    test_df_norm.to_csv(norm_test_path, index=False)
    
    print(f"Node {node} -> Train range: [{train_df_norm[NUMERIC_COLS].min().min():.4f}, {train_df_norm[NUMERIC_COLS].max().max():.4f}] | Test range: [{test_df_norm[NUMERIC_COLS].min().min():.4f}, {test_df_norm[NUMERIC_COLS].max().max():.4f}]")

print(f"\nAll globally normalized datasets successfully saved to: {BASE_DIR}")

=== STEP 1: Fitting Global MinMaxScaler on Combined Training Data of All Nodes ===
Global Scaler successfully fitted and saved to: models/global_minmax_scaler.joblib
Global Min values: {'shunt_voltage': np.float64(417.0), 'bus_voltage_V': np.float64(5.149), 'current_mA': np.float64(416.4949645996094), 'power_mW': np.float64(2160.0)}
Global Max values: {'shunt_voltage': np.float64(1268.8), 'bus_voltage_V': np.float64(6.7717), 'current_mA': np.float64(1288.3), 'power_mW': np.float64(6734.0)}

=== STEP 2: Transforming and Saving Per-Node Splits Using the Global Scaler ===
Node A -> Train range: [0.0000, 0.9217] | Test range: [0.0000, 0.8793]
Node B -> Train range: [0.0000, 1.0000] | Test range: [0.0000, 0.9968]
Node C -> Train range: [0.0000, 0.9985] | Test range: [0.0000, 1.0290]
Node D -> Train range: [0.0000, 1.0000] | Test range: [0.0000, 0.9968]
Node E -> Train range: [0.0000, 0.9968] | Test range: [0.0000, 0.9968]
Node F -> Train range: [0.0000, 0.9968] | Test range: [0.0000, 0.9968

In [5]:
paper = "Zur Sicherstellung der Konsistenz im kollaborativen Netzwerk wurden die kontinuierlichen Sensorvariablen über alle Knoten hinweg mit einem einzigen, globalen MinMaxScaler auf den Wertebereich $[0, 1]$ transformiert. Dieser globale Scaler wurde ausschliesslich auf der aggregierten Trainingsmenge aller Knoten kalibriert, um Data Leakage zu verhindern. Durch diese netzwerkweit einheitliche Normalisierung behalten physikalische Messwerte über alle Systemknoten hinweg ihre absolute statistische Bedeutung, was eine verlässliche Basis für den netzwerkweiten Nachbarschaftskonsens und die verteilte Anomalieerkennung bildet."

print(paper)

Zur Sicherstellung der Konsistenz im kollaborativen Netzwerk wurden die kontinuierlichen Sensorvariablen über alle Knoten hinweg mit einem einzigen, globalen MinMaxScaler auf den Wertebereich $[0, 1]$ transformiert. Dieser globale Scaler wurde ausschliesslich auf der aggregierten Trainingsmenge aller Knoten kalibriert, um Data Leakage zu verhindern. Durch diese netzwerkweit einheitliche Normalisierung behalten physikalische Messwerte über alle Systemknoten hinweg ihre absolute statistische Bedeutung, was eine verlässliche Basis für den netzwerkweiten Nachbarschaftskonsens und die verteilte Anomalieerkennung bildet.
